# Lab - Week2   Rupeshkoram dz93614


## Questions:

1. When do the first and last flights leave each day?

1. When there is a missing value in dep_time then this is an indication of canceled flight. Find the number of cancelled flights for each (carrier, origin) combination.

1. Which carrier has the worst dep_delays?

1. Which plane (tailnum) has the worst on-time record?

1. For each plane, count the number of flights before the first delay of greater than 1 hour.

1. By using the flights data find all flights:

1. Had an arrival delay of two or more hours.

1. Flew to Houston (IAH or HOU)

1. Were operated by American, Delta

1. How many values are missing in dep_time?

1. Sort flight to find fastest flight.

1. Which flights travelled the shortest?

1. Merge `flights` dataframe with `weather` dataframe and investigate if weather has any affect on delays

In [48]:
import pandas as pd

flights = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/nyc_flights.csv')

weather = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/relational_data/nyc_weather.csv')

# example merge:
# flights.merge(weather, on= ['year', 'month', 'day', 'hour', 'origin'])

In [49]:
flights

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336771,2013,9,30,NaN,1455,NaN,NaN,1634,NaN,9E,3393,NaN,JFK,DCA,NaN,213,14,55,2013-09-30T18:00:00Z
336772,2013,9,30,NaN,2200,NaN,NaN,2312,NaN,9E,3525,NaN,LGA,SYR,NaN,198,22,0,2013-10-01T02:00:00Z
336773,2013,9,30,NaN,1210,NaN,NaN,1330,NaN,MQ,3461,N535MQ,LGA,BNA,NaN,764,12,10,2013-09-30T16:00:00Z
336774,2013,9,30,NaN,1159,NaN,NaN,1344,NaN,MQ,3572,N511MQ,LGA,CLE,NaN,419,11,59,2013-09-30T15:00:00Z


1.  When do the first and last flights leave each day?

In [50]:
flights['dep_time'] = pd.to_datetime(flights['dep_time'], format='%H%M', errors='coerce')
first_flights = flights.groupby(['year', 'month', 'day'])['dep_time'].min()
last_flights = flights.groupby(['year', 'month', 'day'])['dep_time'].max()

In [51]:
print(first_flights)
print(last_flights)


year  month  day
2013  1      1     1900-01-01 05:17:00
             2     1900-01-01 04:02:00
             3     1900-01-01 03:02:00
             4     1900-01-01 02:05:00
             5     1900-01-01 01:04:00
                           ...        
      12     27    1900-01-01 04:00:00
             28    1900-01-01 05:06:00
             29    1900-01-01 03:09:00
             30    1900-01-01 02:03:00
             31    1900-01-01 01:03:00
Name: dep_time, Length: 365, dtype: datetime64[ns]
year  month  day
2013  1      1     1900-01-01 23:56:00
             2     1900-01-01 23:54:00
             3     1900-01-01 23:49:00
             4     1900-01-01 23:58:00
             5     1900-01-01 23:57:00
                           ...        
      12     27    1900-01-01 23:51:00
             28    1900-01-01 23:58:00
             29    1900-01-01 23:59:00
             30    1900-01-01 23:56:00
             31    1900-01-01 23:56:00
Name: dep_time, Length: 365, dtype: datetime64[ns]


2. When there is a missing value in dep_time then this is an indication of canceled flight. Find the number of cancelled flights for each (carrier, origin) combination.

In [52]:
cancelled_flights = flights[flights['dep_time'].isna()]
cancelled_c = cancelled_flights.groupby(['carrier', 'origin']).size().reset_index(name='cancelled_count')


In [53]:
cancelled_c

,carrier,origin,cancelled_count
0,9E,EWR,68
1,9E,JFK,808
2,9E,LGA,170
3,AA,EWR,99
4,AA,JFK,142
5,AA,LGA,400
6,AS,EWR,2
7,B6,EWR,77
8,B6,JFK,505
9,B6,LGA,79


3.  Which carrier has the worst dep_delays?

In [37]:
w_c_delay = flights.groupby('carrier')['dep_delay'].mean().idxmax()

In [38]:
w_c_delay 

'F9'

4. Which plane (tailnum) has the worst on-time record?

In [39]:
w_p_delay = flights.groupby('tailnum')['dep_delay'].mean().idxmax()


In [40]:
w_p_delay

'N844MH'

5. For each plane, count the number of flights before the first delay of greater than 1 hour.

In [58]:
flights['dep_time'] = pd.to_datetime(flights['dep_time'], format='%H%M', errors='coerce')
flights = flights.sort_values(by=['tailnum', 'dep_time'])
flights['delay_greater_1hr'] = (flights['dep_delay'] > 60).astype(int)
flights['flights_before_delay'] = flights.groupby('tailnum')['delay_gt_1hr'].cumsum()
flights[['tailnum', 'flights_before_delay']]


,tailnum,flights_before_delay
157799,D942DN,0
254418,D942DN,0
157233,D942DN,0
120316,D942DN,1
128914,N0EGMQ,0
...,...,...
334186,NaN,-1
334868,NaN,-1
335782,NaN,-1
336771,NaN,-1


- 6th & 7th & 8th & 9th

In [25]:
filtered_flights = flights[(flights['arr_delay'] >= 120) & (flights['dest'].isin(['IAH', 'HOU'])) & (flights['carrier'].isin(['AA', 'DL']))]


In [26]:
filtered_flights

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,delay_gt_1hr,flights_before_delay
182284,2013,4,19,1900-01-01 06:06:00,1725,761.0,923.0,2020,783.0,AA,...,N3DGAA,JFK,IAH,222.0,1417,17,25,2013-04-19T21:00:00Z,True,1
185724,2013,4,22,1900-01-01 20:14:00,1725,169.0,2309.0,2020,169.0,AA,...,N3BGAA,JFK,IAH,201.0,1417,17,25,2013-04-22T21:00:00Z,True,3
187632,2013,4,24,1900-01-01 19:02:00,1725,97.0,2257.0,2020,157.0,AA,...,N3KEAA,JFK,IAH,217.0,1417,17,25,2013-04-24T21:00:00Z,True,1
234080,2013,6,13,1900-01-01 18:41:00,1735,66.0,2240.0,2030,130.0,AA,...,N3ECAA,JFK,IAH,195.0,1417,17,35,2013-06-13T21:00:00Z,True,2
247557,2013,6,27,1900-01-01 20:39:00,1735,184.0,2345.0,2030,195.0,AA,...,N3JVAA,JFK,IAH,191.0,1417,17,35,2013-06-27T21:00:00Z,True,4
251253,2013,7,1,1900-01-01 20:58:00,1735,203.0,2355.0,2030,205.0,AA,...,N3HTAA,JFK,IAH,203.0,1417,17,35,2013-07-01T21:00:00Z,True,3
252242,2013,7,2,1900-01-01 20:01:00,1735,146.0,2335.0,2030,185.0,AA,...,N3BRAA,JFK,IAH,205.0,1417,17,35,2013-07-02T21:00:00Z,True,5
258444,2013,7,9,1900-01-01 19:37:00,1735,122.0,2240.0,2030,130.0,AA,...,N3ARAA,JFK,IAH,174.0,1417,17,35,2013-07-09T21:00:00Z,True,5
270890,2013,7,22,1900-01-01 20:18:00,1735,163.0,2344.0,2030,194.0,AA,...,N3CAAA,JFK,IAH,196.0,1417,17,35,2013-07-22T21:00:00Z,True,6
276663,2013,7,28,1900-01-01 19:42:00,1735,127.0,2306.0,2030,156.0,AA,...,N3FYAA,JFK,IAH,200.0,1417,17,35,2013-07-28T21:00:00Z,True,3


10. How many values are missing in dep_time?

In [54]:
missing_dep_time = flights['dep_time'].isna().sum()


In [55]:
missing_dep_time

8512

11. Sort flight to find fastest flight.

In [29]:
fastest_flight = flights.loc[flights['air_time'].idxmax()]


In [30]:
fastest_flight

year                                    2013
month                                      3
day                                       17
dep_time                 1900-01-01 13:37:00
sched_dep_time                          1335
dep_delay                                2.0
arr_time                              1937.0
sched_arr_time                          1836
arr_delay                               61.0
carrier                                   UA
flight                                    15
tailnum                               N77066
origin                                   EWR
dest                                     HNL
air_time                               695.0
distance                                4963
hour                                      13
minute                                    35
time_hour               2013-03-17T17:00:00Z
delay_gt_1hr                           False
flights_before_delay                       1
Name: 151467, dtype: object

12. Which flights travelled the shortest?

In [45]:
shortest_flights = flights.nsmallest(5, 'distance') 


In [46]:
shortest_flights

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,delay_gt_1hr,flights_before_delay
275945,2013,7,27,NaT,106,NaN,NaN,245,NaN,US,...,NaN,EWR,LGA,NaN,17,1,6,2013-07-27T05:00:00Z,False,207
2658,2013,1,3,1900-01-01 21:27:00,2129,-2.0,2222.0,2224,-2.0,EV,...,N13989,EWR,PHL,30.0,80,21,29,2013-01-04T02:00:00Z,False,0
3083,2013,1,4,1900-01-01 12:40:00,1200,40.0,1333.0,1306,27.0,EV,...,N14972,EWR,PHL,30.0,80,12,0,2013-01-04T17:00:00Z,False,1
3426,2013,1,4,1900-01-01 18:29:00,1615,134.0,1937.0,1721,136.0,EV,...,N15983,EWR,PHL,28.0,80,16,15,2013-01-04T21:00:00Z,True,1
3578,2013,1,4,1900-01-01 21:28:00,2129,-1.0,2218.0,2224,-6.0,EV,...,N27962,EWR,PHL,32.0,80,21,29,2013-01-05T02:00:00Z,False,0


13. Merge flights dataframe with weather dataframe and investigate if weather has any affect on delays

In [56]:
merged_data = flights.merge(weather, on=['year', 'month', 'day', 'hour', 'origin'])


In [57]:
merged_data

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour_y
0,2013,1,1,1900-01-01 05:17:00,515,2.0,830.0,819,11.0,UA,...,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z
1,2013,1,1,1900-01-01 05:54:00,558,-4.0,740.0,728,12.0,UA,...,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z
2,2013,1,1,1900-01-01 05:33:00,529,4.0,850.0,830,20.0,UA,...,39.92,24.98,54.81,250.0,14.96014,21.86482,0.0,1011.4,10.0,2013-01-01T10:00:00Z
3,2013,1,1,1900-01-01 05:42:00,540,2.0,923.0,850,33.0,AA,...,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0,2013-01-01T10:00:00Z
4,2013,1,1,1900-01-01 05:44:00,545,-1.0,1004.0,1022,-18.0,B6,...,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0,2013-01-01T10:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335215,2013,9,30,1900-01-01 22:40:00,2245,-5.0,2334.0,2351,-17.0,B6,...,60.98,55.94,83.47,230.0,9.20624,NaN,0.0,1016.5,10.0,2013-10-01T02:00:00Z
335216,2013,9,30,1900-01-01 22:40:00,2250,-10.0,2347.0,7,-20.0,B6,...,60.98,55.94,83.47,230.0,9.20624,NaN,0.0,1016.5,10.0,2013-10-01T02:00:00Z
335217,2013,9,30,1900-01-01 22:41:00,2246,-5.0,2345.0,1,-16.0,B6,...,60.98,55.94,83.47,230.0,9.20624,NaN,0.0,1016.5,10.0,2013-10-01T02:00:00Z
335218,2013,9,30,1900-01-01 23:07:00,2255,12.0,2359.0,2358,1.0,B6,...,60.98,55.94,83.47,230.0,9.20624,NaN,0.0,1016.5,10.0,2013-10-01T02:00:00Z
